In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
from pathlib import Path
from tqdm import tqdm
import json
import joblib
import logging
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

In [ ]:
CONFIG = {
    'architecture': 'resnet50',
    'pretrained': True,
    'image_size': 224,
    
    'batch_size': 32,
    'num_epochs_stage1': 10, 
    'num_epochs_stage2': 15,
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    
    'max_samples_per_class': 2000,
    'food_not_food_ratio': 1.0,
    
    'confidence_threshold': 0.7,
    
    'early_stopping_patience': 5,
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
DATASET_ROOT = Path(r"d:/sugarcare_backend/data/your_datasets/food_image_recogination")
TRAIN_IMAGES_DIR = DATASET_ROOT / "train_images"
TRAIN_CSV = DATASET_ROOT / "train_img.csv"
OUTPUT_DIR = Path(r"d:/sugarcare_backend/models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATASET_ROOT}")
print(f"Train CSV: {TRAIN_CSV}")
print(f"Train Images: {TRAIN_IMAGES_DIR}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
print("\n" + "="*60)
print("LOADING DATASET")
print("="*60)

df = pd.read_csv(TRAIN_CSV)
print(f"\nTotal images: {len(df):,}")
print(f"Columns: {list(df.columns)}")

cols_lower = [c.lower() for c in df.columns]
file_col = df.columns[cols_lower.index('imageid') if 'imageid' in cols_lower else 0]
label_col = df.columns[cols_lower.index('classname') if 'classname' in cols_lower else 1]

print(f"\nFile column: {file_col}")
print(f"Label column: {label_col}")

df[label_col] = df[label_col].astype(str).str.strip()

class_counts = df[label_col].value_counts()
print(f"\nUnique classes: {len(class_counts)}")

not_food_variants = ['not_food', 'notfood', 'not-food', 'background']
not_food_mask = df[label_col].str.lower().isin(not_food_variants)
not_food_count = not_food_mask.sum()
food_count = len(df) - not_food_count

print(f"\nFood images: {food_count:,}")
print(f"Not-food images: {not_food_count:,}")

print(f"\nTop 10 classes:")
for cls, count in class_counts.head(10).items():
    print(f"  {cls}: {count:,}")

In [ ]:
print("\n" + "="*60)
print("DATA PREPROCESSING")
print("="*60)

from concurrent.futures import ThreadPoolExecutor
import os

def validate_image(row):
    """Check if image file exists and is readable."""
    img_path = TRAIN_IMAGES_DIR / row[file_col]
    if not img_path.exists():
        return None
    try:
        with Image.open(img_path) as img:
            img.verify() 
        return row.name
    except Exception:
        return None

print("\nValidating images (this may take a moment)...")

sample_size = min(1000, len(df))
sample_df = df.sample(n=sample_size, random_state=42)

valid_count = 0
invalid_files = []

for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Validating sample"):
    img_path = TRAIN_IMAGES_DIR / row[file_col]
    if img_path.exists():
        try:
            with Image.open(img_path) as img:
                img.load() 
            valid_count += 1
        except Exception as e:
            invalid_files.append(row[file_col])
    else:
        invalid_files.append(row[file_col])

valid_rate = valid_count / sample_size * 100
print(f"\nSample validation results:")
print(f"  Valid images: {valid_count}/{sample_size} ({valid_rate:.1f}%)")
print(f"  Invalid/missing: {len(invalid_files)}")

if invalid_files:
    print(f"\nExample invalid files: {invalid_files[:5]}")

if invalid_files:
    df = df[~df[file_col].isin(invalid_files)].reset_index(drop=True)
    print(f"\nFiltered dataset: {len(df):,} images remaining")

print("\n" + "-"*40)
print("Class Distribution Summary:")
print("-"*40)
food_classes = df[~df[label_col].str.lower().isin(['not_food', 'notfood'])][label_col].nunique()
print(f"  Food classes: {food_classes}")
print(f"  Total food images: {len(df[~df[label_col].str.lower().isin(['not_food', 'notfood'])]):,}")
print(f"  Total not_food images: {len(df[df[label_col].str.lower().isin(['not_food', 'notfood'])]):,}")
print(f"  Total dataset size: {len(df):,}")

In [ ]:
print("\n" + "="*60)
print("DATA AUGMENTATION")
print("="*60)

train_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'] + 32, CONFIG['image_size'] + 32)),
    transforms.RandomCrop(CONFIG['image_size']),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
])

val_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Train augmentations:")
print("  - Random crop, flip, rotation")
print("  - Color jitter, grayscale")
print("  - Random erasing")
print("\nValidation: Resize only")

In [ ]:

class FoodBinaryDataset(Dataset):
    """Stage 1: Food vs Not-Food Binary Classification"""
    
    def __init__(self, df, images_dir, file_col, label_col, transform=None, balance=True, max_per_class=None):
        self.images_dir = Path(images_dir)
        self.transform = transform
        
        not_food_variants = ['not_food', 'notfood', 'not-food', 'background']
        df = df.copy()
        df['is_food'] = ~df[label_col].str.lower().isin(not_food_variants)
        df['binary_label'] = df['is_food'].astype(int)
        
        if balance:
            food_df = df[df['binary_label'] == 1]
            not_food_df = df[df['binary_label'] == 0]
            
            min_count = min(len(food_df), len(not_food_df))
            if max_per_class:
                min_count = min(min_count, max_per_class)
            
            food_df = food_df.sample(n=min_count, random_state=42)
            not_food_df = not_food_df.sample(n=min_count, random_state=42)
            df = pd.concat([food_df, not_food_df]).reset_index(drop=True)
            print(f"Balanced: {len(food_df)} food + {len(not_food_df)} not_food = {len(df)} total")
        
        self.files = df[file_col].tolist()
        self.labels = df['binary_label'].tolist()
        self.class_to_idx = {'not_food': 0, 'food': 1}
        self.idx_to_class = {0: 'not_food', 1: 'food'}
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        img_path = self.images_dir / self.files[idx]
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            img = Image.new('RGB', (224, 224), color=(128, 128, 128))
        
        if self.transform:
            img = self.transform(img)
        
        return img, self.labels[idx]


class FoodTypeDataset(Dataset):
    """Stage 2: Food Type Multi-class Classification (food images only)"""
    
    def __init__(self, df, images_dir, file_col, label_col, transform=None, max_per_class=None):
        self.images_dir = Path(images_dir)
        self.transform = transform
        
        not_food_variants = ['not_food', 'notfood', 'not-food', 'background']
        df = df[~df[label_col].str.lower().isin(not_food_variants)].copy()
        
        if max_per_class:
            balanced_dfs = []
            for cls in df[label_col].unique():
                cls_df = df[df[label_col] == cls]
                if len(cls_df) > max_per_class:
                    cls_df = cls_df.sample(n=max_per_class, random_state=42)
                balanced_dfs.append(cls_df)
            df = pd.concat(balanced_dfs).reset_index(drop=True)
        
        # Create class mapping
        classes = sorted(df[label_col].unique())
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.idx_to_class = {i: c for c, i in self.class_to_idx.items()}
        
        self.files = df[file_col].tolist()
        self.labels = [self.class_to_idx[c] for c in df[label_col].tolist()]
        
        print(f"Food type dataset: {len(self.files)} images, {len(classes)} classes")
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        img_path = self.images_dir / self.files[idx]
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            img = Image.new('RGB', (224, 224), color=(128, 128, 128))
        
        if self.transform:
            img = self.transform(img)
        
        return img, self.labels[idx]

print("Dataset classes defined:")
print("  - FoodBinaryDataset: Food vs Not-Food")
print("  - FoodTypeDataset: Food Type Classification")

In [ ]:
def build_model(num_classes, architecture='resnet50', pretrained=True):
    """Build classification model with specified architecture."""
    
    if architecture == 'resnet50':
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        model = models.resnet50(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )
        
    elif architecture == 'efficientnet':
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )
        
    elif architecture == 'mobilenet':
        weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        model = models.mobilenet_v3_small(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        
    else:
        raise ValueError(f"Unknown architecture: {architecture}")
    
    return model.to(device)

print(f"Model builder ready. Architecture: {CONFIG['architecture']}")

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs, lr, stage_name, save_path):
    """Train model with early stopping and metrics logging."""
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=CONFIG['weight_decay'])
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    
    use_amp = torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler() if use_amp else None
    
    best_val_acc = 0.0
    epochs_no_improve = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    print(f"\n{'='*60}")
    print(f"TRAINING: {stage_name}")
    print(f"{'='*60}")
    print(f"Epochs: {num_epochs}, LR: {lr}, Batch: {train_loader.batch_size}")
    
    for epoch in range(1, num_epochs + 1):
        # Training phase
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs} [Train]")
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad(set_to_none=True)
            
            if use_amp:
                with torch.cuda.amp.autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
            
            train_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            train_correct += preds.eq(labels).sum().item()
            train_total += labels.size(0)
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{train_correct/train_total:.4f}'})
        
        train_loss /= train_total
        train_acc = train_correct / train_total
        
        # Validation phase
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Epoch {epoch}/{num_epochs} [Val]", leave=False):
                images, labels = images.to(device), labels.to(device)
                
                if use_amp:
                    with torch.cuda.amp.autocast():
                        outputs = model(images)
                        loss = criterion(outputs, labels)
                else:
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                _, preds = outputs.max(1)
                val_correct += preds.eq(labels).sum().item()
                val_total += labels.size(0)
        
        val_loss /= val_total
        val_acc = val_correct / val_total
        
        # Update scheduler
        scheduler.step(val_acc)
        
        # Log metrics
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
              f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            torch.save({
                'state_dict': model.state_dict(),
                'val_acc': val_acc,
                'epoch': epoch,
            }, save_path)
            print(f"  ✓ Best model saved (val_acc={val_acc:.4f})")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= CONFIG['early_stopping_patience']:
                print(f"  Early stopping at epoch {epoch}")
                break
    
    print(f"\nBest validation accuracy: {best_val_acc:.4f}")
    return history, best_val_acc

print("Training function ready.")

In [ ]:
print("\n" + "="*60)
print("STAGE 1: FOOD vs NOT-FOOD")
print("="*60)

binary_dataset = FoodBinaryDataset(
    df, TRAIN_IMAGES_DIR, file_col, label_col,
    transform=train_transform,
    balance=True,
    max_per_class=CONFIG['max_samples_per_class'] * 3 
)

indices = list(range(len(binary_dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.15, random_state=42)

class DatasetSubset(Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, label = self.dataset[self.indices[i]]
        return img, label

train_binary = DatasetSubset(binary_dataset, train_idx)
val_binary = DatasetSubset(binary_dataset, val_idx)

train_loader_s1 = DataLoader(train_binary, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0, pin_memory=True)
val_loader_s1 = DataLoader(val_binary, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0, pin_memory=True)

print(f"\nTrain: {len(train_binary)} samples")
print(f"Val: {len(val_binary)} samples")
print(f"Classes: {binary_dataset.class_to_idx}")

In [ ]:
print("\nBuilding Stage 1 model...")
model_stage1 = build_model(
    num_classes=2, 
    architecture=CONFIG['architecture'],
    pretrained=CONFIG['pretrained']
)

history_s1, best_acc_s1 = train_model(
    model_stage1,
    train_loader_s1,
    val_loader_s1,
    num_epochs=CONFIG['num_epochs_stage1'],
    lr=CONFIG['learning_rate'],
    stage_name="Stage 1: Food vs Not-Food",
    save_path=OUTPUT_DIR / "food_binary_model.pth"
)

In [ ]:
print("\n" + "="*60)
print("STAGE 2: FOOD TYPE CLASSIFICATION")
print("="*60)

food_dataset = FoodTypeDataset(
    df, TRAIN_IMAGES_DIR, file_col, label_col,
    transform=train_transform,
    max_per_class=CONFIG['max_samples_per_class']
)

indices = list(range(len(food_dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.15, random_state=42, stratify=[food_dataset.labels[i] for i in indices])

train_food = DatasetSubset(food_dataset, train_idx)
val_food = DatasetSubset(food_dataset, val_idx)

train_loader_s2 = DataLoader(train_food, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0, pin_memory=True)
val_loader_s2 = DataLoader(val_food, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0, pin_memory=True)

num_food_classes = len(food_dataset.class_to_idx)
print(f"\nTrain: {len(train_food)} samples")
print(f"Val: {len(val_food)} samples")
print(f"Food classes: {num_food_classes}")
print(f"\nSample classes: {list(food_dataset.class_to_idx.keys())[:10]}...")

In [ ]:
print("\nBuilding Stage 2 model...")
model_stage2 = build_model(
    num_classes=num_food_classes,
    architecture=CONFIG['architecture'],
    pretrained=CONFIG['pretrained']
)

history_s2, best_acc_s2 = train_model(
    model_stage2,
    train_loader_s2,
    val_loader_s2,
    num_epochs=CONFIG['num_epochs_stage2'],
    lr=CONFIG['learning_rate'],
    stage_name="Stage 2: Food Type Classification",
    save_path=OUTPUT_DIR / "food_type_model.pth"
)

In [ ]:
print("\n" + "="*60)
print("EXPORTING COMBINED MODEL")
print("="*60)

checkpoint_s1 = torch.load(OUTPUT_DIR / "food_binary_model.pth", map_location=device)
checkpoint_s2 = torch.load(OUTPUT_DIR / "food_type_model.pth", map_location=device)

combined_export = {
    'stage1_state_dict': checkpoint_s1['state_dict'],
    'stage1_class_to_idx': binary_dataset.class_to_idx,
    'stage1_idx_to_class': binary_dataset.idx_to_class,
    'stage1_val_acc': checkpoint_s1['val_acc'],
    
    'stage2_state_dict': checkpoint_s2['state_dict'],
    'stage2_class_to_idx': food_dataset.class_to_idx,
    'stage2_idx_to_class': food_dataset.idx_to_class,
    'stage2_val_acc': checkpoint_s2['val_acc'],
    
    'architecture': CONFIG['architecture'],
    'image_size': CONFIG['image_size'],
    'confidence_threshold': CONFIG['confidence_threshold'],
    'num_food_classes': num_food_classes,
    
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225],
}

output_path = OUTPUT_DIR / "food_recognition_trained.pkl"
joblib.dump(combined_export, output_path)

print(f"\nCombined model exported: {output_path}")
print(f"\nModel Summary:")
print(f"  Architecture: {CONFIG['architecture']}")
print(f"  Stage 1 (Food/Not-Food) Accuracy: {checkpoint_s1['val_acc']:.4f}")
print(f"  Stage 2 (Food Types) Accuracy: {checkpoint_s2['val_acc']:.4f}")
print(f"  Food Classes: {num_food_classes}")
print(f"  Confidence Threshold: {CONFIG['confidence_threshold']}")

In [ ]:
print("\n" + "="*60)
print("TEST INFERENCE")
print("="*60)

def predict_food(image_path, model_s1, model_s2, class_to_idx_s2, idx_to_class_s2, threshold=0.7):
    """Two-stage food prediction with confidence."""
    
    img = Image.open(image_path).convert('RGB')
    img_tensor = val_transform(img).unsqueeze(0).to(device)
    
    model_s1.eval()
    with torch.no_grad():
        output_s1 = model_s1(img_tensor)
        probs_s1 = torch.softmax(output_s1, dim=1)[0]
        is_food_prob = probs_s1[1].item() 
    
    if is_food_prob < threshold:
        return {
            'is_food': False,
            'food_confidence': is_food_prob,
            'food_type': None,
            'type_confidence': None,
            'message': 'Image does not appear to contain food'
        }
    
    model_s2.eval()
    with torch.no_grad():
        output_s2 = model_s2(img_tensor)
        probs_s2 = torch.softmax(output_s2, dim=1)[0]
        top_prob, top_idx = probs_s2.max(0)
        top_prob = top_prob.item()
        food_type = idx_to_class_s2[top_idx.item()]
    
    if top_prob < threshold:
        return {
            'is_food': True,
            'food_confidence': is_food_prob,
            'food_type': food_type,
            'type_confidence': top_prob,
            'message': f'Food detected but type uncertain (confidence: {top_prob:.2f})'
        }
    
    return {
        'is_food': True,
        'food_confidence': is_food_prob,
        'food_type': food_type,
        'type_confidence': top_prob,
        'message': f'Detected: {food_type}'
    }

model_s1_test = build_model(2, CONFIG['architecture'], False)
model_s1_test.load_state_dict(checkpoint_s1['state_dict'])
model_s1_test.eval()

model_s2_test = build_model(num_food_classes, CONFIG['architecture'], False)
model_s2_test.load_state_dict(checkpoint_s2['state_dict'])
model_s2_test.eval()

test_images = list(TRAIN_IMAGES_DIR.glob("*.jpg"))[:5]
print(f"\nTesting on {len(test_images)} sample images:\n")

for img_path in test_images:
    result = predict_food(
        img_path, 
        model_s1_test, 
        model_s2_test, 
        food_dataset.class_to_idx, 
        food_dataset.idx_to_class,
        threshold=CONFIG['confidence_threshold']
    )
    print(f"{img_path.name}:")
    print(f"  Is Food: {result['is_food']} ({result['food_confidence']:.2f})")
    if result['food_type']:
        print(f"  Type: {result['food_type']} ({result['type_confidence']:.2f})")
    print(f"  {result['message']}\n")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(history_s1['train_loss'], label='Train Loss')
axes[0, 0].plot(history_s1['val_loss'], label='Val Loss')
axes[0, 0].set_title('Stage 1: Loss (Food vs Not-Food)')
axes[0, 0].legend()
axes[0, 0].set_xlabel('Epoch')

axes[0, 1].plot(history_s1['train_acc'], label='Train Acc')
axes[0, 1].plot(history_s1['val_acc'], label='Val Acc')
axes[0, 1].set_title('Stage 1: Accuracy (Food vs Not-Food)')
axes[0, 1].legend()
axes[0, 1].set_xlabel('Epoch')

axes[1, 0].plot(history_s2['train_loss'], label='Train Loss')
axes[1, 0].plot(history_s2['val_loss'], label='Val Loss')
axes[1, 0].set_title('Stage 2: Loss (Food Type)')
axes[1, 0].legend()
axes[1, 0].set_xlabel('Epoch')

axes[1, 1].plot(history_s2['train_acc'], label='Train Acc')
axes[1, 1].plot(history_s2['val_acc'], label='Val Acc')
axes[1, 1].set_title('Stage 2: Accuracy (Food Type)')
axes[1, 1].legend()
axes[1, 1].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_history.png', dpi=150)
plt.show()

print(f"\nTraining history saved to: {OUTPUT_DIR / 'training_history.png'}")

In [ ]:
print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)

print(f"""
Model Architecture: {CONFIG['architecture']}

STAGE 1 - Food Detection:
  - Classes: food, not_food
  - Best Accuracy: {best_acc_s1:.4f}
  - Detects: humans, animals, objects as 'not_food'

STAGE 2 - Food Classification:
  - Classes: {num_food_classes} food types
  - Best Accuracy: {best_acc_s2:.4f}

Confidence Threshold: {CONFIG['confidence_threshold']}
  - Predictions below this threshold are rejected

Exported Files:
  - {OUTPUT_DIR / 'food_recognition_trained.pkl'}
  - {OUTPUT_DIR / 'food_binary_model.pth'}
  - {OUTPUT_DIR / 'food_type_model.pth'}
  - {OUTPUT_DIR / 'training_history.png'}

Usage:
  1. Load image
  2. Stage 1: Check if food (threshold: {CONFIG['confidence_threshold']})
  3. If food → Stage 2: Classify food type
  4. Return result with confidence scores
""")